# 07 - Đánh giá các mô hình RELEASE (giai đoạn phát hành)
UIT-ViQuAD 2.0, trích xuất câu trả lời. Notebook đánh giá các mô hình **release**: huấn luyện lại các công thức đã đóng băng (stage 3) nhưng **có lưu trọng số** (fp16 trong `outputs/release/`). Logits/summary đọc từ `outputs/sweep/release/`.

## 0. Mục đích và giao thức
- Release = huấn luyện lại đúng công thức đóng băng của stage 3, cùng dữ liệu (`split_ids.json`), cùng metric `src/metrics.py`, cùng biến thể hậu xử lý `min/64/20` (không tune lại lưới). Ngưỡng no-answer tune trên **dev** cho từng run và cho từng ensemble.
- Mọi lựa chọn (thành viên ensemble, mô hình đơn tốt nhất) dựa **chỉ trên dev hoặc log huấn luyện**, không dựa test. Các mô hình release được chấm trên **test đúng một lần** (`RUN_TEST`).
- **Kết quả báo cáo chính vẫn là kết quả gốc của stage 3 / notebook 05.** Notebook này chỉ đánh giá các trọng số được phát hành và so sánh chúng với kết quả gốc.

In [1]:
import sys, json, logging, contextlib, io, itertools
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from src import evalkit as ek
from src.postprocess import apply_threshold, tune_threshold
logging.basicConfig(level=logging.WARNING)
pd.set_option("display.width", 200)

RUN_TEST = False  # scored once on 2026-11-14; outputs below are from that run
N_BOOT = 2000
MODELS = ["xlmr_base", "phobert", "xlmr_large"]
VARIANT = dict(zip(("null_variant", "max_answer_length", "n_best"), ek.DEFAULT_VARIANT))  # min/64/20, đóng băng
assert tuple(VARIANT.values()) == ("min", 64, 20), VARIANT
print("RUN_TEST =", RUN_TEST, "| variant", VARIANT)

RUN_TEST = True | variant {'null_variant': 'min', 'max_answer_length': 64, 'n_best': 20}


## 1. Bảng các run release

In [ ]:
allruns = ek.load_runs(ROOT / "outputs")
runs = allruns[allruns.stage == "release"].sort_values(["model", "seed"]).reset_index(drop=True)
assert len(runs) == 10, len(runs)
cols = ["model", "run_name", "seed", "lr", "epochs", "dev_f1", "dev_em", "dev_hasans_f1", "dev_noans_acc", "threshold", "dev_f1_at0", "train_runtime_s", "est_units"]
display(runs[cols].round(2))

,model,run_name,seed,lr,epochs,dev_f1,dev_em,dev_hasans_f1,dev_noans_acc,threshold,dev_f1_at0,train_runtime_s,est_units
0,phobert,phobert__release_s42,42.0,0.0,5.0,68.33,52.68,69.40,66.14,-2.00,66.38,293.20,None
1,phobert,phobert__release_s43,43.0,0.0,5.0,66.28,50.38,70.79,57.02,-1.50,64.58,294.57,None
2,phobert,phobert__release_s44,44.0,0.0,5.0,62.78,47.18,66.86,54.40,-1.00,62.57,295.20,None
3,phobert,phobert__release_s45,45.0,0.0,5.0,60.86,43.13,74.38,33.12,0.00,60.86,292.61,None
4,xlmr_base,xlmr_base__release_s42,42.0,0.0,5.0,61.53,47.97,63.87,56.71,0.50,60.46,435.15,None
5,xlmr_base,xlmr_base__release_s43,43.0,0.0,5.0,63.96,49.86,64.89,62.05,0.50,62.80,431.84,None
6,xlmr_base,xlmr_base__release_s44,44.0,0.0,5.0,60.32,45.67,67.24,46.12,0.75,59.75,430.45,None
7,xlmr_large,xlmr_large__release_s42,42.0,0.0,3.0,77.76,62.40,77.65,77.99,-1.50,76.87,775.17,None
8,xlmr_large,xlmr_large__release_s43,43.0,0.0,3.0,77.20,61.98,76.62,78.41,-0.75,77.00,775.04,None
9,xlmr_large,xlmr_large__release_s44,44.0,0.0,3.0,77.51,62.53,78.20,76.10,1.75,76.73,774.96,None


## 2. Dev F1 từng seed: release so với stage 3, và độ ổn định huấn luyện

In [3]:
s3 = allruns[allruns.stage == "stage3"][["model", "seed", "dev_f1"]].rename(columns={"dev_f1": "stage3_dev_f1"})
cmp_ = runs[["model", "seed", "dev_f1"]].rename(columns={"dev_f1": "release_dev_f1"}).merge(s3, on=["model", "seed"], how="left")
cmp_["delta"] = cmp_.release_dev_f1 - cmp_.stage3_dev_f1
display(cmp_.round(2))

rows = []
for _, r in runs.iterrows():
    h = json.loads((Path(r.run_dir) / "summary.json").read_text())["log_history"]
    ev = [x for x in h if "eval_loss" in x]
    gn = max((x for x in h if "grad_norm" in x), key=lambda x: x["grad_norm"])
    rows.append({"run": r.run_name, "dev_f1": r.dev_f1, "threshold": r.threshold,
                 "best_epoch": json.loads((Path(r.run_dir) / "summary.json").read_text()).get("best_epoch"),
                 "eval_loss_per_epoch": [round(x["eval_loss"], 3) for x in ev],
                 "max_grad_norm": round(gn["grad_norm"], 1), "at_step": gn["step"]})
stab = pd.DataFrame(rows).set_index("run")
display(stab.round(2))

,model,seed,release_dev_f1,stage3_dev_f1,delta
0,phobert,42.0,68.33,68.31,0.02
1,phobert,43.0,66.28,66.45,-0.17
2,phobert,44.0,62.78,68.41,-5.63
3,phobert,45.0,60.86,NaN,NaN
4,xlmr_base,42.0,61.53,62.30,-0.77
5,xlmr_base,43.0,63.96,63.06,0.90
6,xlmr_base,44.0,60.32,59.69,0.63
7,xlmr_large,42.0,77.76,76.92,0.84
8,xlmr_large,43.0,77.20,77.13,0.08
9,xlmr_large,44.0,77.51,77.55,-0.04


,dev_f1,threshold,best_epoch,eval_loss_per_epoch,max_grad_norm,at_step
run,,,,,,
phobert__release_s42,68.33,-2.00,4.0,"[1.353, 1.159, 1.122, 1.112, 1.123]",22.4,550
phobert__release_s43,66.28,-1.50,3.0,"[1.353, 1.217, 1.144, 1.155, 2.173]",254.8,4400
phobert__release_s44,62.78,-1.00,2.0,"[1.431, 1.182, 2.722, 2.648, 2.649]",80.9,2850
phobert__release_s45,60.86,0.00,2.0,"[1.354, 1.206, 2.696, 2.655, 2.679]",115.0,2450
xlmr_base__release_s42,61.53,0.50,4.0,"[1.494, 1.312, 1.269, 1.257, 1.286]",35.8,150
xlmr_base__release_s43,63.96,0.50,4.0,"[1.454, 1.308, 1.244, 1.23, 1.256]",33.2,250
xlmr_base__release_s44,60.32,0.75,3.0,"[1.495, 1.313, 1.262, 1.283, 1.287]",41.4,2150
xlmr_large__release_s42,77.76,-1.50,2.0,"[1.029, 0.898, 0.918]",207.6,150
xlmr_large__release_s43,77.20,-0.75,2.0,"[1.007, 0.901, 0.937]",127.5,2550


**Nhận xét về độ ổn định (PhoBERT).** Với `xlmr_base` và `xlmr_large`, F1 dev của release sát stage 3 (xem cột `delta`). Với PhoBERT, hai trong bốn seed release (**s44, s45**) bị **phân kỳ**: `eval_loss` tăng từ khoảng 1,2 lên khoảng 2,7 ở epoch 3 và không hồi phục; checkpoint tốt nhất được khôi phục từ epoch 2 nên F1 dev chỉ 62,78 và 60,86 (so với 68,33 / 66,28 ở s42 / s43). Seed s43 cũng có `eval_loss` tăng vọt ở epoch 5 (2,173) nhưng checkpoint tốt nhất (epoch 3) được giữ. Ở stage 3 (bản gốc), cùng công thức cho PhoBERT dev 68,31 / 66,45 / 68,41 và không phân kỳ. **Nguyên nhân khác biệt chưa được xác lập** (có thể do tính không tất định của huấn luyện, thay đổi môi trường Colab, hoặc điều khác; chưa kiểm chứng).

**Quyết định (chốt, người dùng):** ensemble release của PhoBERT chỉ gồm **seed 42 và 43**. Seed 44 và 45 được giữ làm bằng chứng nhưng bị loại, dựa trên **dev và log huấn luyện, không dựa test**. Hệ quả: ensemble PhoBERT chỉ có 2 thành viên.

## 3. Định nghĩa ensemble release

In [4]:
ENS = {"xlmr_base": [42, 43, 44], "phobert": [42, 43], "xlmr_large": [42, 43, 44]}
RUNDIR = {r.run_name: r.run_dir for r in runs.itertuples()}
ens_runs = {m: [f"{m}__release_s{s}" for s in ENS[m]] for m in MODELS}
assert all(n in RUNDIR for v in ens_runs.values() for n in v)
display(pd.Series({m: ", ".join(v) for m, v in ens_runs.items()}, name="thành viên ensemble"))

xlmr_base     xlmr_base__release_s42, xlmr_base__release_s43...
phobert              phobert__release_s42, phobert__release_s43
xlmr_large    xlmr_large__release_s42, xlmr_large__release_s...
Name: thành viên ensemble, dtype: str

## 4. Dev: ensemble (logits trung bình) với biến thể `min/64/20`
Biến thể đóng băng, không tune lại lưới. Ngưỡng tune trên dev cho từng ensemble; từng run dùng ngưỡng dev của riêng nó (`dev_threshold.json`). Điểm dev hơi lạc quan vì ngưỡng tune trên chính dev.

In [5]:
def thr_of(run_name):
    return json.loads((Path(RUNDIR[run_name]) / "dev_threshold.json").read_text())["threshold"]

ctx, loaded, pe_dev, thr_ens, thr_run, dev_rows = {}, {}, {}, {}, {}, []
for m in MODELS:
    names = sorted(runs[runs.model == m].run_name)          # tất cả run, kể cả run bị loại
    loaded[m] = {n: ek.load_run(RUNDIR[n]) for n in names}
    r0 = loaded[m][ens_runs[m][0]]
    ex, feats, extract = ek.build_eval_context(m, "dev", expected_hash=r0["dev"]["feature_hash"])
    ctx[m] = (ex, feats, extract)
    refs = ek.refs_of(ex)
    def ev(s, e, thr):
        sc = ek.rescore(ex, feats, s, e, extract, **VARIANT)
        return sc, ek.per_example(apply_threshold(sc, thr), refs)
    for n in names:
        thr_run[n] = thr_of(n)
        _, pe = ev(loaded[m][n]["dev"]["start"], loaded[m][n]["dev"]["end"], thr_run[n])
        pe_dev[n] = pe
    s, e, _, _ = ek.ensemble_logits([loaded[m][n] for n in ens_runs[m]])
    sc = ek.rescore(ex, feats, s, e, extract, **VARIANT)
    thr_ens[m], _ = tune_threshold(sc, refs)
    pe_dev[m + " ensemble"] = ek.per_example(apply_threshold(sc, thr_ens[m]), refs)
    f = {n: pe_dev[n].f1.mean() * 100 for n in ens_runs[m]}
    dev_rows.append({"model": m, "members_f1": {n.split("_")[-1]: round(v, 2) for n, v in f.items()},
                     "member_mean": np.mean(list(f.values())), "ensemble_f1": pe_dev[m + " ensemble"].f1.mean() * 100,
                     "gain_vs_mean": pe_dev[m + " ensemble"].f1.mean() * 100 - np.mean(list(f.values())),
                     "ens_threshold": thr_ens[m]})
display(pd.DataFrame(dev_rows).set_index("model").round(2))
print("Từng run release (cả run bị loại), ngưỡng dev của run:")
display(pd.DataFrame({"dev_f1_rescored": {n: pe_dev[n].f1.mean() * 100 for n in thr_run}, "threshold": thr_run}).round(2))

D:\00_Workspace\AI_Lab\RAG_official\Extractive_QA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Casting the dataset:   0%|          | 0/25533 [00:00<?, ? examples/s]


Casting the dataset: 100%|██████████| 25533/25533 [00:00<00:00, 773733.04 examples/s]


Casting the dataset:   0%|          | 0/2912 [00:00<?, ? examples/s]


Casting the dataset: 100%|██████████| 2912/2912 [00:00<00:00, 727142.54 examples/s]


Casting the dataset:   0%|          | 0/3814 [00:00<?, ? examples/s]

... (5 outputs left out)


Map: 100%|██████████| 2912/2912 [00:00<00:00, 4288.31 examples/s]

,members_f1,member_mean,ensemble_f1,gain_vs_mean,ens_threshold
model,,,,,
xlmr_base,"{'s42': 61.53, 's43': 63.99, 's44': 60.36}",61.96,63.33,1.37,1.00
phobert,"{'s42': 68.33, 's43': 66.28}",67.31,68.19,0.88,-1.50
xlmr_large,"{'s42': 77.75, 's43': 77.19, 's44': 77.5}",77.48,78.59,1.11,-0.25


Từng run release (cả run bị loại), ngưỡng dev của run:


,dev_f1_rescored,threshold
xlmr_base__release_s42,61.53,0.50
xlmr_base__release_s43,63.99,0.50
xlmr_base__release_s44,60.36,0.75
phobert__release_s42,68.33,-2.00
phobert__release_s43,66.28,-1.50
phobert__release_s44,62.78,-1.00
phobert__release_s45,60.86,0.00
xlmr_large__release_s42,77.75,-1.50
xlmr_large__release_s43,77.19,-0.75
xlmr_large__release_s44,77.50,1.75


In [6]:
ci_dev = pd.concat({k: ek.ci_table(p, N_BOOT) for k, p in pe_dev.items() if k.endswith("ensemble")})
lab = {("overall", "em"): "EM", ("overall", "f1"): "F1", ("HasAns", "f1"): "HasAns F1", ("NoAns", "em"): "NoAns acc"}
def pivot(ci):
    t = ci.reset_index(level=0).rename(columns={"level_0": "model"})
    t["k"] = [lab.get((a, b)) for a, b in zip(t.subset, t.metric)]
    t = t.dropna(subset=["k"])
    t["val"] = t.apply(lambda r: f"{r['mean']:.2f} [{r['lo']:.2f}, {r['hi']:.2f}]", axis=1)
    return t.pivot(index="model", columns="k", values="val")[["EM", "F1", "HasAns F1", "NoAns acc"]]
print("DEV, ensemble release, CI bootstrap 95% (N_BOOT =", N_BOOT, ")")
display(pivot(ci_dev))

DEV, ensemble release, CI bootstrap 95% (N_BOOT = 2000 )


k,EM,F1,HasAns F1,NoAns acc
model,,,,
phobert ensemble,"52.23 [50.34, 54.12]","68.19 [66.64, 69.70]","71.84 [70.15, 73.48]","60.69 [57.75, 63.73]"
xlmr_base ensemble,"49.18 [47.32, 51.00]","63.33 [61.75, 64.92]","68.76 [67.04, 70.45]","52.20 [49.06, 55.56]"
xlmr_large ensemble,"63.70 [61.88, 65.49]","78.59 [77.21, 79.90]","78.33 [76.87, 79.79]","79.14 [76.52, 81.66]"


## 5. Mô hình đơn tốt nhất mỗi loại (chọn theo F1 dev trong các thành viên ensemble)
Ghi `outputs/release/best_models.json`. Ngưỡng của mô hình đơn lấy từ `dev_threshold.json` của run; ngưỡng ensemble tune trên dev.

In [7]:
best = {}
for m in MODELS:
    sub = runs[runs.run_name.isin(ens_runs[m])].sort_values("dev_f1", ascending=False)
    b = sub.iloc[0]
    best[m] = {"run": b.run_name, "dir": f"outputs/release/{m}/{b.run_name}", "threshold": thr_of(b.run_name), "dev_f1": float(b.dev_f1)}
    assert (ROOT / best[m]["dir"] / "model.safetensors").exists()
best["ensembles"] = {**ens_runs, "ensemble_thresholds": {m: float(thr_ens[m]) for m in MODELS}}
out = ROOT / "outputs" / "release" / "best_models.json"
out.write_text(json.dumps(best, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(best, indent=2))

{
  "xlmr_base": {
    "run": "xlmr_base__release_s43",
    "dir": "outputs/release/xlmr_base/xlmr_base__release_s43",
    "threshold": 0.5,
    "dev_f1": 63.95969567614896
  },
  "phobert": {
    "run": "phobert__release_s42",
    "dir": "outputs/release/phobert/phobert__release_s42",
    "threshold": -2.0,
    "dev_f1": 68.33245444454916
  },
  "xlmr_large": {
    "run": "xlmr_large__release_s42",
    "dir": "outputs/release/xlmr_large/xlmr_large__release_s42",
    "threshold": -1.5,
    "dev_f1": 77.76031400226918
  },
  "ensembles": {
    "xlmr_base": [
      "xlmr_base__release_s42",
      "xlmr_base__release_s43",
      "xlmr_base__release_s44"
    ],
    "phobert": [
      "phobert__release_s42",
      "phobert__release_s43"
    ],
    "xlmr_large": [
      "xlmr_large__release_s42",
      "xlmr_large__release_s43",
      "xlmr_large__release_s44"
    ],
    "ensemble_thresholds": {
      "xlmr_base": 1.0,
      "phobert": -1.5,
      "xlmr_large": -0.25
    }
  }
}


## 6. TEST (chấm một lần)
Ensemble release mỗi mô hình với ngưỡng dev của ensemble, từng thành viên với ngưỡng dev của run, và mô hình đơn tốt nhất. Cùng ô này đọc lại logits **stage 3 gốc** (run đóng băng, chỉ đọc) và tính lại ensemble gốc cho mục 7 (ngưỡng dev của ensemble gốc tune trên dev stage 3, giống notebook 05).

In [8]:
assert isinstance(RUN_TEST, bool)
assert all(m in thr_ens for m in MODELS) and all(n in thr_run for v in ens_runs.values() for n in v), "ngưỡng dev chưa nạp"
pe_test, pe_orig = {}, {}
s3runs = allruns[allruns.stage == "stage3"]
if not RUN_TEST:
    print("RUN_TEST=False: không chấm test.")
else:
    buf = io.StringIO()
    prev = logging.root.manager.disable
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(buf):
            for m in MODELS:
                ex, feats, extract = ctx[m]  # ngữ cảnh dev cho dev stage 3; test dựng riêng bên dưới
                names = ens_runs[m]
                rs = {n: ek.load_run(RUNDIR[n], allow_test=True) for n in names}
                exT, featsT, extractT = ek.build_eval_context(m, "test", allow_test=True, expected_hash=rs[names[0]]["test"]["feature_hash"])
                refsT = ek.refs_of(exT)
                def score(s, e, thr):
                    sc = ek.rescore(exT, featsT, s, e, extractT, **VARIANT)
                    return ek.per_example(apply_threshold(sc, thr), refsT)
                s, e, _, _ = ek.ensemble_logits(list(rs.values()), split="test", allow_test=True)
                pe_test[f"{m} ensemble"] = score(s, e, thr_ens[m])
                for n in names:
                    pe_test[n] = score(rs[n]["test"]["start"], rs[n]["test"]["end"], thr_run[n])
                pe_test[f"{m} best single ({best[m]['run']})"] = pe_test[best[m]["run"]]
                # ensemble gốc stage 3 (chỉ đọc): ngưỡng tune trên dev stage 3
                o = [ek.load_run(d, allow_test=True) for d in s3runs[s3runs.model == m].sort_values("seed").run_dir]
                assert o[0]["test"]["feature_hash"] == rs[names[0]]["test"]["feature_hash"]
                sd, ed, _, _ = ek.ensemble_logits(o)
                refsD = ek.refs_of(ex)
                t0, _ = tune_threshold(ek.rescore(ex, feats, sd, ed, extract, **VARIANT), refsD)
                st, et, _, _ = ek.ensemble_logits(o, split="test", allow_test=True)
                pe_orig[m] = score(st, et, t0)
    finally:
        logging.disable(prev)
    print("WARNING: TEST SPLIT SCORED")
    for m in MODELS:
        print(f"{m}: release ensemble F1 {pe_test[m + ' ensemble'].f1.mean()*100:.2f} | original (recomputed) ensemble F1 {pe_orig[m].f1.mean()*100:.2f}")


Casting the dataset:   0%|          | 0/25533 [00:00<?, ? examples/s]


Casting the dataset: 100%|██████████| 25533/25533 [00:00<00:00, 783618.07 examples/s]


Casting the dataset:   0%|          | 0/2912 [00:00<?, ? examples/s]


Casting the dataset: 100%|██████████| 2912/2912 [00:00<00:00, 562226.72 examples/s]


Casting the dataset:   0%|          | 0/3814 [00:00<?, ? examples/s]


Casting the dataset: 100%|██████████| 3814/3814 [00:00<00:00, 567413.03 examples/s]

... (3 outputs left out)


Map:  79%|███████▊  | 3000/3814 [00:00<00:00, 4459.93 examples/s]


Map: 100%|██████████| 3814/3814 [00:00<00:00, 4816.39 examples/s]


Map: 100%|██████████| 3814/3814 [00:00<00:00, 4792.57 examples/s]

xlmr_base: release ensemble F1 63.59 | original (recomputed) ensemble F1 62.40
phobert: release ensemble F1 68.62 | original (recomputed) ensemble F1 68.97
xlmr_large: release ensemble F1 77.68 | original (recomputed) ensemble F1 77.89


## 6b. Bảng test với CI bootstrap 95% (EM, F1, HasAns F1, NoAns acc)

In [9]:
if pe_test:
    ci_test = pd.concat({k: ek.ci_table(p, N_BOOT) for k, p in pe_test.items()})
    display(pivot(ci_test))
else:
    print("Chưa có điểm test.")

k,EM,F1,HasAns F1,NoAns acc
model,,,,
phobert best single (phobert__release_s42),"58.18 [56.61, 59.81]","68.10 [66.67, 69.52]","69.00 [67.44, 70.57]","66.06 [63.39, 68.73]"
phobert ensemble,"59.07 [57.55, 60.62]","68.62 [67.22, 70.01]","71.58 [70.04, 73.10]","61.84 [59.00, 64.51]"
phobert__release_s42,"58.18 [56.61, 59.81]","68.10 [66.67, 69.52]","69.00 [67.44, 70.57]","66.06 [63.39, 68.73]"
phobert__release_s43,"57.29 [55.74, 58.84]","66.82 [65.37, 68.29]","69.56 [68.00, 71.11]","60.55 [57.80, 63.39]"
xlmr_base best single (xlmr_base__release_s43),"54.33 [52.73, 55.98]","63.14 [61.64, 64.66]","63.48 [61.83, 65.09]","62.36 [59.52, 65.12]"
xlmr_base ensemble,"54.46 [52.81, 56.06]","63.59 [62.11, 65.00]","67.40 [65.74, 68.99]","54.87 [52.02, 57.54]"
xlmr_base__release_s42,"52.99 [51.36, 54.64]","61.28 [59.72, 62.81]","61.78 [60.10, 63.48]","60.12 [57.28, 62.96]"
xlmr_base__release_s43,"54.33 [52.73, 55.98]","63.14 [61.64, 64.66]","63.48 [61.83, 65.09]","62.36 [59.52, 65.12]"
xlmr_base__release_s44,"51.05 [49.45, 52.60]","61.10 [59.54, 62.51]","66.80 [65.25, 68.33]","48.06 [45.13, 50.90]"


## 7. So sánh với ensemble GỐC của stage 3 (test F1)
Số gốc lấy từ notebook 05 (cell 19, bảng test với CI): xlmr_base 62.40, phobert 68.97, xlmr_large 77.89. Ô trên cũng tính lại ensemble gốc từ logits stage 3 để kiểm tra chéo và để làm kiểm định ghép cặp.

In [10]:
ORIG_TEST_F1 = {"xlmr_base": 62.40, "phobert": 68.97, "xlmr_large": 77.89}  # nb05 cell 19 (ensemble 3 seed, stage 3)
if pe_test:
    rows, prs = [], []
    for m in MODELS:
        rel = pe_test[m + " ensemble"].f1.mean() * 100
        rows.append({"model": m, "original_ens_f1": ORIG_TEST_F1[m], "recomputed_orig": pe_orig[m].f1.mean() * 100,
                     "release_ens_f1": rel, "delta(release-orig)": rel - ORIG_TEST_F1[m], "release_members": len(ens_runs[m])})
        r = ek.paired_bootstrap(pe_test[m + " ensemble"].f1.values * 100, pe_orig[m].f1.values * 100, N_BOOT)
        prs.append({"model": m, "A": "release ens", "B": "original ens", **r})
    display(pd.DataFrame(rows).set_index("model").round(2))
    print("Kiểm định ghép cặp release vs gốc (p = tỉ lệ resample có hiệu A-B <= 0):")
    display(pd.DataFrame(prs).round(4))
else:
    print("Chưa có điểm test.")

,original_ens_f1,recomputed_orig,release_ens_f1,delta(release-orig),release_members
model,,,,,
xlmr_base,62.40,62.40,63.59,1.19,3
phobert,68.97,68.97,68.62,-0.35,2
xlmr_large,77.89,77.89,77.68,-0.21,3


Kiểm định ghép cặp release vs gốc (p = tỉ lệ resample có hiệu A-B <= 0):


,model,A,B,diff,lo,hi,p
0,xlmr_base,release ens,original ens,1.1901,0.4400,1.8949,0.000
1,phobert,release ens,original ens,-0.3495,-0.8789,0.1704,0.901
2,xlmr_large,release ens,original ens,-0.2084,-0.8610,0.4585,0.741


## 8. Tài nguyên

In [11]:
rows = []
for m in MODELS:
    rj = [json.loads(l) for l in (ROOT / "outputs/sweep/release" / m / "results.jsonl").read_text().splitlines() if l.strip()]
    sizes = {n: (ROOT / "outputs/release" / m / n / "model.safetensors").stat().st_size / 2**20 for n in runs[runs.model == m].run_name}
    rows.append({"model": m, "params_M": ek.PARAMS_M.get(m), "n_runs_trained": len(rj), "weights_MiB_per_run(fp16)": np.mean(list(sizes.values())),
                 "ensemble_weights_MiB": sum(sizes[n] for n in ens_runs[m]), "est_units_total": sum(x.get("est_units") or 0 for x in rj),
                 "train_runtime_s_total": sum(x.get("train_runtime_s") or 0 for x in rj)})
res = pd.DataFrame(rows).set_index("model")
display(res.round(1))
print("Tổng est_units của 10 run release:", round(res.est_units_total.sum(), 2))

,params_M,n_runs_trained,weights_MiB_per_run(fp16),ensemble_weights_MiB,est_units_total,train_runtime_s_total
model,,,,,,
xlmr_base,277.5,3,529.2,1587.7,2.6,1297.4
phobert,135.0,4,256.4,512.8,2.5,1175.6
xlmr_large,559.9,3,1066.0,3197.9,4.6,2325.2


Tổng est_units của 10 run release: 9.71


## 9. Hạn chế
- Trọng số release lưu ở fp16; logits lúc suy luận từ trọng số fp16 có thể lệch logits lúc huấn luyện (kiểm tra trước đó: khớp argmax 99,0-100% trên 100-300 ví dụ dev), nên điểm có thể lệch nhẹ so với `summary.json`.
- Huấn luyện không hoàn toàn tất định; cùng công thức và seed vẫn có thể cho kết quả khác stage 3 (xem mục 2).
- PhoBERT dễ vỡ ổn định: hai trong bốn seed release phân kỳ; nguyên nhân chưa được xác lập.
- Ensemble release của PhoBERT chỉ có 2 thành viên (s42, s43), ít hơn 3 seed của stage 3, nên không so sánh trực tiếp được hoàn toàn.
- Ngưỡng và biến thể tune trên dev nên điểm dev lạc quan; số công bố chính là test của stage 3 gốc, còn số release là kiểm chứng trọng số phát hành. CI bootstrap chỉ phản ánh phương sai do mẫu ví dụ, không phải phương sai huấn luyện.